# RAGFlow 检索

根据 **question** 调用 RAGFlow `POST /api/v1/retrieval`，从知识库取回相似 chunk。

与本 Notebook **放在同一目录** 的 `.env` 中配置：

- `RAGFLOW_API_KEY`
- `RAGFLOW_DATASET_ID`
- `RAGFLOW_RETRIEVAL_URL`（完整 URL），或 `RAGFLOW_ADDRESS`（形如 `host:port`，将自动拼为 `http://{address}/api/v1/retrieval`）

打开 Notebook 时请将工作目录设为该文件夹（Jupyter 默认即为 notebook 所在目录）。

In [2]:
from __future__ import annotations

import os
import time
from pathlib import Path

import requests

WORKDIR = Path.cwd().resolve()
ENV_PATH = WORKDIR / ".env"
if not ENV_PATH.is_file():
    raise FileNotFoundError(
        f"未找到 {ENV_PATH}。请将 .env 与本 ipynb 放在同一目录后再运行。"
    )

for line in ENV_PATH.read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if not line or line.startswith("#") or "=" not in line:
        continue
    key, value = line.split("=", 1)
    os.environ[key.strip()] = value.strip().strip('"').strip("'")

print(f"已加载: {ENV_PATH.name}")

API_KEY = os.environ["RAGFLOW_API_KEY"].strip()
DATASET_ID = os.environ["RAGFLOW_DATASET_ID"].strip()

RETRIEVAL_URL = os.environ.get("RAGFLOW_RETRIEVAL_URL", "").strip().strip('"').strip("'")
if not RETRIEVAL_URL:
    address = os.environ.get("RAGFLOW_ADDRESS", "").strip()
    if not address:
        raise RuntimeError(".env 需设置 RAGFLOW_RETRIEVAL_URL 或 RAGFLOW_ADDRESS")
    RETRIEVAL_URL = f"http://{address}/api/v1/retrieval"

print(f"Retrieval URL: {RETRIEVAL_URL}")
print(f"Dataset ID: {DATASET_ID}")

已加载: .env
Retrieval URL: http://42.51.100.38:9380/api/v1/retrieval
Dataset ID: <RAGFLOW_DATASET_ID>


In [3]:
# 检索问句（按需修改）
QUESTION = (
    "HumanEval Qiskit: 使用 Statevector 得到 Bell 态向量，"
    "与 quantum_info API 写法"
)

PAGE_SIZE = 5
SIMILARITY_THRESHOLD = 0.2
TIMEOUT_SECONDS = 120

In [4]:
def ragflow_retrieve(
    question: str,
    *,
    dataset_id: str,
    retrieval_url: str,
    api_key: str,
    page_size: int = PAGE_SIZE,
    similarity_threshold: float = SIMILARITY_THRESHOLD,
    timeout: float = TIMEOUT_SECONDS,
) -> tuple[dict, float]:
    """POST /api/v1/retrieval，返回 (响应 JSON, 耗时秒)。"""
    payload = {
        "question": question,
        "dataset_ids": [dataset_id],
        "page_size": page_size,
        "similarity_threshold": similarity_threshold,
    }
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}",
    }
    t0 = time.perf_counter()
    resp = requests.post(
        retrieval_url, headers=headers, json=payload, timeout=timeout
    )
    wall = time.perf_counter() - t0
    resp.raise_for_status()
    data = resp.json()
    if data.get("code") != 0:
        raise RuntimeError(data.get("message") or data)
    return data, wall


def parse_chunks(response: dict) -> list[dict]:
    rows = []
    for ch in (response.get("data") or {}).get("chunks") or []:
        if not isinstance(ch, dict):
            continue
        content = (ch.get("content") or ch.get("text") or "").strip()
        if not content:
            continue
        rows.append(
            {
                "keyword": ch.get("document_keyword") or ch.get("doc_name") or "",
                "similarity": float(ch.get("similarity") or 0.0),
                "content": content,
            }
        )
    return rows


raw, elapsed = ragflow_retrieve(
    QUESTION,
    dataset_id=DATASET_ID,
    retrieval_url=RETRIEVAL_URL,
    api_key=API_KEY,
)
chunks = parse_chunks(raw)

print(f"question 长度: {len(QUESTION)} | 耗时: {elapsed:.2f}s | chunk 数: {len(chunks)}")
for i, ch in enumerate(chunks, start=1):
    preview = ch["content"][:240].replace("\n", " ")
    print(f"\n--- Chunk {i} | {ch['keyword']} | sim={ch['similarity']:.4f} ---\n{preview}…")

question 长度: 66 | 耗时: 4.15s | chunk 数: 5

--- Chunk 1 | qiskit_circuit_library_data_preparation_state_preparation_Statevector_from_instruction.md | sim=0.6100 ---
# Qiskit 2.4.1 API: `qiskit.circuit.library.data_preparation.state_preparation.Statevector.from_instruction` ## 基本信息 - Framework: Qiskit - Version: 2.4.1 - Module: `qiskit.circuit.library.data_preparation.state_preparation` - API: `qiskit.c…

--- Chunk 2 | qiskit_circuit_library_data_preparation_state_preparation_Statevector_purity.md | sim=0.5835 ---
# Qiskit 2.4.1 API: `qiskit.circuit.library.data_preparation.state_preparation.Statevector.purity` ## 基本信息 - Framework: Qiskit - Version: 2.4.1 - Module: `qiskit.circuit.library.data_preparation.state_preparation` - API: `qiskit.circuit.lib…

--- Chunk 3 | qiskit_circuit_library_data_preparation_state_preparation_Statevector_copy.md | sim=0.5808 ---
# Qiskit 2.4.1 API: `qiskit.circuit.library.data_preparation.state_preparation.Statevector.copy` ## 基本信息 - Framework: Qiskit - Vers